In [1]:
# Import the expression finder from our new analyzer
from analyzer.expression_finder import print_expression_analysis

In [2]:
# Analyze the sample file
print_expression_analysis("sample_files/sample.py")

Analyzing expressions in: sample_files/sample.py
Found 39 expression nodes:
------------------------------------------------------------
 1. Line  3 | Attribute    | attr='name'
 2. Line  3 | Name         | name='self'
 3. Line  3 | Name         | name='name'
 4. Line  4 | Attribute    | attr='profile'
 5. Line  4 | Name         | name='self'
 6. Line  4 | Call         | function call
 7. Line  4 | Name         | name='Profile'
 8. Line  7 | Attribute    | attr='status'
 9. Line  7 | Attribute    | attr='profile'
10. Line  7 | Name         | name='self'
11. Line 11 | Attribute    | attr='status'
12. Line 11 | Name         | name='self'
13. Line 11 | Constant     | value='active'
14. Line 14 | Dict         | other
15. Line 14 | Constant     | value='status'
16. Line 14 | Constant     | value='info'
17. Line 14 | Attribute    | attr='status'
18. Line 14 | Name         | name='self'
19. Line 14 | Constant     | value='user profile'
20. Line 17 | Name         | name='user'
21. Line 17 | Ca

In [10]:
# Demo of the new hierarchical graph system
from analyzer.graph import (
    CodebaseGraph, ProjectNode, ModuleNode, ClassNode, FunctionNode,
    HasModuleEdge, HasClassEdge, HasFunctionEdge
)

# Create a graph for our project
graph = CodebaseGraph("atlas_project")

# Create some nodes (now with cleaner API)
project = ProjectNode(id="atlas_project", name="atlas_project")
module = ModuleNode(id="atlas_project.sample", name="sample", path="sample_files/sample.py")
user_class = ClassNode(id="atlas_project.sample.User", name="User", line_number=1)
init_method = FunctionNode(
    id="atlas_project.sample.User.__init__", 
    name="__init__", 
    line_number=2, 
    arguments=["self", "name"],
    is_method=True
)

# Add nodes to graph
graph.add_node(project)
graph.add_node(module)
graph.add_node(user_class)
graph.add_node(init_method)

# Create relationships
graph.add_edge(HasModuleEdge(source_id="atlas_project", target_id="atlas_project.sample"))
graph.add_edge(HasClassEdge(source_id="atlas_project.sample", target_id="atlas_project.sample.User"))
graph.add_edge(HasFunctionEdge(source_id="atlas_project.sample.User", target_id="atlas_project.sample.User.__init__"))

# Demo the functionality
print("=== Graph Stats ===")
stats = graph.get_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

print("\n=== Node Identity Demo ===")
# Create duplicate node with same id (no metadata parameter!)
user_class2 = ClassNode(id="atlas_project.sample.User", name="User", line_number=999)

print(f"Same hash: {hash(user_class) == hash(user_class2)}")
print(f"Same equality: {user_class == user_class2}")
print(f"Set deduplication: {len({user_class, user_class2})} items")

print(f"Different metadata: {user_class.metadata != user_class2.metadata}")

print("\n=== Graph Traversal Demo ===")
# Get modules connected to project
modules = graph.get_connected_nodes("atlas_project", HasModuleEdge)
print(f"Project has {len(modules)} modules:")
for mod in modules:
    print(f"  - {mod.name} at {mod.path}")

# Get classes in the module
classes = graph.get_connected_nodes("atlas_project.sample", HasClassEdge)
print(f"Module has {len(classes)} classes:")
for cls in classes:
    print(f"  - {cls.name} (line {cls.line_number})")

=== Graph Stats ===
total_nodes: 4
total_edges: 3
ProjectNode_count: 1
ModuleNode_count: 1
ClassNode_count: 1
FunctionNode_count: 1
HasModuleEdge_count: 1
HasClassEdge_count: 1
HasFunctionEdge_count: 1

=== Node Identity Demo ===
Same hash: True
Same equality: True
Set deduplication: 1 items
Different metadata: True

=== Graph Traversal Demo ===
Project has 1 modules:
  - sample at sample_files/sample.py
Module has 1 classes:
  - User (line 1)
